In [ ]:
from pprint import pprint

from src.orchestration.orch import Orch
from src.validation.cross_validation import CrossValidator

train_data_path = "data/GenePT_gene_embedding_ada_text.pickle"
target_data_path = "data/AdamsonWeissman2016_GSM2406681_10X010.h5ad"

In [ ]:
# 1. Holdout Validation

holdout = Orch(
    train_data_path,
    target_data_path
)

holdout_report = holdout.run()

In [ ]:
# 2. K-fold cross validation

ridge = Orch(
    train_data_path,
    target_data_path,
    model="ridge",
    validation_method="cross_validation",
    n_splits=5,
    random_state=42
)

ridge_report = ridge.run()

In [ ]:
pprint(ridge_report["summary"])

In [ ]:
# 3. shared folds across models

cv = CrossValidator(
    trainer=ridge.trainer,
    n_splits=5,
    random_state=42
)

shared_folds = list(cv.split_folds(ridge.X_processed))

knn = Orch(
    train_data_path,
    target_data_path,
    model="knn",
    validation_method="cross_validation"
)

knn_report = knn.run(folds=shared_folds)

In [ ]:
for ridge_fold, knn_fold in zip(
    ridge_report["folds"],
    knn_report["folds"]
):
    assert ridge_fold["train_ids"] == knn_fold["train_ids"]
    assert ridge_fold["test_ids"] == knn_fold["test_ids"]

print("Both models used identical folds.")

In [ ]:
# 4. custom model hyperparameters

ridge_custom = Orch(
    train_data_path,
    target_data_path,
    model="ridge",
    params={"alpha": 10.0},
    validation_method="cross_validation"
)

ridge_custom_report = ridge_custom.run(folds=shared_folds)

In [ ]:
# 5. example evaluation results

reports = {
    "Ridge (alpha=1.0)": ridge_report,
    "Ridge (alpha=10.0)": ridge_custom_report,
    "KNN": knn_report
}

for name, report in reports.items():
    result = report["summary"]["test"]["pearson_delta"]

    print(
        f"{name}: "
        f"{result['mean']:.4f} ± {result['std']:.4f}"
    )

## Notes

- Shared folds require identical perturbation ordering across experiments.
- Data preparation is performed before validation.
- Cross-validation does not train a final model on the complete dataset.
- The reported standard deviation represents variability across folds.
- Model comparison and ranking are handled separately by the Evaluation module.